# Notebook 5 — Confronto Finale e Analisi

Obiettivo: valutare entrambi i modelli su tutti i test set, produrre la tabella comparativa con RMSE/MAE/NASA Score e generare i grafici di analisi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import sys
sys.path.append('..')
from src.metrics import evaluate, evaluate_by_rul_range
# Importo le classi custom del Transformer: necessarie perché Keras le deve
# riconoscere quando carica il file .keras dal disco.
# Il decorator @register_keras_serializable nel sorgente le registra globalmente
# appena vengono importate — non serve passarle manualmente a load_model.
from src.models.transformer_model import PositionalEncoding, TransformerEncoderBlock

DATA_DIR   = '../data/processed'
MODEL_DIR  = '../saved_models'
RESULT_DIR = '../experiments/results'
PLOT_DIR   = '../plots'

import os
os.makedirs(RESULT_DIR, exist_ok=True)

## Le tre metriche di valutazione

| Metrica | Formula | Interpretazione |
|---------|---------|----------------|
| **RMSE** | √mean((y-ŷ)²) | Penalizza errori grandi quadraticamente. Metrica principale. |
| **MAE** | mean(|y-ŷ|) | Errore medio assoluto. Più robusto agli outlier, più interpretabile. |
| **NASA Score** | Σ(exp asimmetrico) | **Asimmetrico**: penalizza la predizione tardiva più dell'anticipata. |

Il NASA Score è il più importante dal punto di vista applicativo:
- **Predizione tardiva** (ŷ > y, sovrastima RUL): il motore è più vicino al guasto di quanto pensiamo → rischio reale
- **Predizione anticipata** (ŷ < y): interveniamo prima del necessario → manutenzione inutile ma motore salvo

La penalità è `exp(d/10) - 1` per d > 0 e `exp(-d/13) - 1` per d < 0 → curve con pendenze diverse.

In [ ]:
# custom_objects necessario per caricare il Transformer (contiene layer custom)
# Per LSTM non serve: usa solo layer standard di Keras
custom_objects = {
    'PositionalEncoding': PositionalEncoding,
    'TransformerEncoderBlock': TransformerEncoderBlock,
}

rows = []
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    X_test = np.load(f'{DATA_DIR}/X_test_{fd}.npy')
    y_test = np.load(f'{DATA_DIR}/y_test_{fd}.npy')

    for arch in ['lstm', 'transformer']:
        # Carico il miglior checkpoint salvato da ModelCheckpoint (non i pesi finali)
        model = tf.keras.models.load_model(
            f'{MODEL_DIR}/{arch}_{fd}.keras',
            custom_objects=custom_objects if arch == 'transformer' else None
        )
        # Inferenza sull'intero test set in batch → output shape (n_motori, 1)
        y_pred = model.predict(X_test, verbose=0).flatten()  # → shape (n_motori,)
        m = evaluate(y_test, y_pred)
        rows.append({'model': arch.upper(), 'dataset': fd, **m})
        print(f'{arch.upper()} {fd}: RMSE={m["rmse"]:.2f}, MAE={m["mae"]:.2f}, Score={m["nasa_score"]:.0f}')

results_df = pd.DataFrame(rows)
results_df.to_csv(f'{RESULT_DIR}/main_comparison.csv', index=False)
results_df

## Scatter Plot: Predetto vs Reale

**Come leggere il grafico:**
- **Diagonale rossa** = predizione perfetta (ŷ = y)
- **Punti sopra la diagonale** = predizione tardiva (ŷ > y, sovrastima RUL) → NASA Score penalizza fortemente
- **Punti sotto la diagonale** = predizione anticipata (ŷ < y) → penalità minore
- Dispersione attorno alla diagonale → RMSE/MAE più alto

Nota FD004-Transformer: diversi punti con y ≈ 0–30 ma ŷ ≈ 80–120 → predizioni tardive gravi
che spiegano il NASA Score di 2.369.435.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for col, fd in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
    X_test = np.load(f'{DATA_DIR}/X_test_{fd}.npy')
    y_test = np.load(f'{DATA_DIR}/y_test_{fd}.npy')
    for row, arch in enumerate(['lstm', 'transformer']):
        model = tf.keras.models.load_model(
            f'{MODEL_DIR}/{arch}_{fd}.keras',
            custom_objects=custom_objects if arch == 'transformer' else None
        )
        y_pred = model.predict(X_test, verbose=0).flatten()
        ax = axes[row, col]
        ax.scatter(y_test, y_pred, alpha=0.5, s=10)
        # Linea rossa = predizione perfetta: punti lontani = errori grandi
        ax.plot([0, 125], [0, 125], 'r--')
        ax.set_title(f'{arch.upper()} — {fd}')
        ax.set_xlabel('True RUL')
        ax.set_ylabel('Predicted RUL')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/05_scatter_pred_vs_true.png', dpi=150)
plt.show()

## Analisi per Fascia di RUL

Divido il test set in tre fasce per capire **dove** il modello sbaglia di più:

- **RUL 0–50** (fase critica): motori vicini al guasto. Il segnale dei sensori è forte e monotono → errori minori
- **RUL 50–100** (fase di transizione): il degrado è in corso ma il segnale è ancora ambiguo → **errori più alti**
- **RUL 100–125** (fase sana, cappata): tutti trattati come RUL=125, la variabilità vera è > 125 → errori intermedi

Questa analisi risponde alla domanda: "I modelli sono più affidabili quando il motore sta per guastarsi?"

In [ ]:
range_rows = []
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    X_test = np.load(f'{DATA_DIR}/X_test_{fd}.npy')
    y_test = np.load(f'{DATA_DIR}/y_test_{fd}.npy')
    for arch in ['lstm', 'transformer']:
        model = tf.keras.models.load_model(
            f'{MODEL_DIR}/{arch}_{fd}.keras',
            custom_objects=custom_objects if arch == 'transformer' else None
        )
        y_pred = model.predict(X_test, verbose=0).flatten()
        # evaluate_by_rul_range: calcola MAE e RMSE separatamente per 0-50, 50-100, 100-125
        by_range = evaluate_by_rul_range(y_test, y_pred)
        for rng, m in by_range.items():
            range_rows.append({'model': arch.upper(), 'dataset': fd, 'rul_range': rng, **m})

range_df = pd.DataFrame(range_rows)
range_df.to_csv(f'{RESULT_DIR}/rul_range_analysis.csv', index=False)
range_df